In [ ]:
#Import packages 

import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import pandas as pd 
from shapely.geometry import shape 
import json 
from shapely import wkt 
from shapely.geometry import Point
from shapely.geometry import box
from math import cos, radians
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.cm import ScalarMappable
import seaborn as sns 
import matplotlib
from statsmodels.tsa.seasonal import seasonal_decompose
import contextily as ctx
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from sklearn.cluster import KMeans

import glob
import os
import csv
import ast
import matplotlib.patheffects as path_effects
from matplotlib.ticker import PercentFormatter

from libpysal import weights

### Read in source files 

In [ ]:
# study area gdfs 
new_ea = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/new_ea.json')
new_data_study_area = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/new_data_study_area.json')

# PQR sites 
new_sites = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/new_sites.json')

# shapefile (all EAs)
filt_all_eas_dem_df = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/filt_all_eas_dem_df.json')

### Mapping of EAs and PQR sites 

In [ ]:
ea_copy_pqr = new_ea.copy()
ea_copy_pqr = ea_copy_pqr[['ea_code9ch', 'geometry']]
merged_sites_pqr = gpd.sjoin(new_sites, ea_copy_pqr, how = 'inner', predicate = 'intersects')

merged_sites_pqr = merged_sites_pqr[['space_grouping', 'geometry', 'ea_code9ch']].reset_index(drop=True)

grouped_sites_df = merged_sites_pqr.groupby('ea_code9ch').agg({
    'space_grouping': lambda x: list(set(x)),  
})

grouped_sites_df = grouped_sites_df.rename(columns = {'space_grouping':'site_id'})

### 2021 Census Districts 

In [ ]:
# read districts shapefile for plotting 
dist_2021 = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/district_2021_census.geojson')

In [ ]:
### Filter out Study Area Districts 

dist_2021 = gpd.sjoin(
    dist_2021, merged_sites_pqr.to_crs('EPSG:4326'), 
    how="inner", 
    predicate='contains' 
).reset_index(drop=True).drop(columns=['index_right'])

dist_2021 = dist_2021.drop_duplicates(['Name']).reset_index(drop=True)

# reproject to local CRS 
dist_2021_transformed = dist_2021.to_crs(merged_sites_pqr.crs)

### Revised EA shapefile  

In [ ]:
filt_all_eas_dem_df_filt = gpd.sjoin(
    filt_all_eas_dem_df, dist_2021_transformed.drop(columns = ['ea_code9ch']), 
    how="inner", 
    predicate='intersects' 
).reset_index(drop=True)

## exclude additional EAs around the edges for basemap plotting 
additional_eas_to_exclude = [31400149, 31400142, 31400139, 31400134, 31400133, 30801173, 30801139, 30801138, 30801136, 30801063, 30801062, 30100656, 30100652, 30200407, 30200368, 30801041, 30801001, 30100689, 30200426]

filt_all_eas_dem_df_filt = filt_all_eas_dem_df_filt[
    ~filt_all_eas_dem_df_filt['ea_code9ch'].isin(additional_eas_to_exclude)
]

## Outages 

In [ ]:
## 2022 
geo_pqr_22 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/2022/merged_outage_n_voltage_hourly_22_NEW.csv')
geo_pqr_22 = geo_pqr_22.drop(columns = ['Unnamed: 0'])
geo_pqr_22 = geo_pqr_22[['time', 'site_id', 'outage_events', 'outage_mins']]

## 2023 
geo_pqr_23 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/2023/merged_outage_n_voltage_hourly_23_NEW.csv')
geo_pqr_23 = geo_pqr_23.drop(columns = ['Unnamed: 0'])
geo_pqr_23 = geo_pqr_23[['time', 'site_id', 'outage_events', 'outage_mins']]

In [ ]:
## concatenate 
geo_pqr_all = pd.concat([geo_pqr_22, geo_pqr_23], ignore_index=True)
geo_pqr_all['time'] = pd.to_datetime(geo_pqr_all['time'])
geo_pqr_all = geo_pqr_all[~(geo_pqr_all['site_id'] == 0)]

### 1+ hour outages 

In [ ]:
dur = 60

# filter out outages with duration >= dur 
geo_pqr_all['outage_mins_dur'] = (
    np.floor(geo_pqr_all['outage_mins'] / dur)
).where(geo_pqr_all['outage_mins'] >= dur, 0)


# Set outage_events to 0 where outage_hours_dur is 0
geo_pqr_all['outage_events_adj'] = geo_pqr_all['outage_events'].where(
    geo_pqr_all['outage_mins_dur'] > 0, 0
)

geo_dur_grouped = geo_pqr_all.groupby('site_id')[['outage_events_adj', 'outage_mins_dur']].sum().reset_index()

## 60min / 1hr 
geo_dur_grouped = geo_dur_grouped.rename(columns = {'outage_events_adj':'60min_outage_events'})

## remove site '0'
geo_dur_grouped = geo_dur_grouped[~(geo_dur_grouped['site_id'] == 0)]

##
geo_1hr_grouped = geo_dur_grouped.copy()

### 8+ hour 

In [ ]:
dur = 480

# filter out outages with duration >= dur 
geo_pqr_all['outage_mins_dur'] = (
    np.floor(geo_pqr_all['outage_mins'] / dur)
).where(geo_pqr_all['outage_mins'] >= dur, 0)


# Set outage_events to 0 where outage_hours_dur is 0
geo_pqr_all['outage_events_adj'] = geo_pqr_all['outage_events'].where(
    geo_pqr_all['outage_mins_dur'] > 0, 0
)

geo_dur_grouped = geo_pqr_all.groupby('site_id')[['outage_events_adj', 'outage_mins_dur']].sum().reset_index()

## 480min / 8hr 
geo_dur_grouped = geo_dur_grouped.rename(columns = {'outage_events_adj':'480min_outage_events'})

## remove site '0'
geo_dur_grouped = geo_dur_grouped[~(geo_dur_grouped['site_id'] == 0)]

##
geo_8hr_grouped = geo_dur_grouped.copy()

### Undervolts - 60_min 

In [ ]:
## 2022 
geo_pqr_22 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/Geospatial_Voltage_Durations/Files/voltage_hourly_22_und_60.csv')
geo_pqr_22 = geo_pqr_22.drop(columns = ['Unnamed: 0'])
geo_pqr_22_uv = geo_pqr_22[['time', 'site_id', 'total_undervolt_events', 'total_undervolt_duration']]

## 2023 
geo_pqr_23 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/Geospatial_Voltage_Durations/Files/voltage_hourly_23_und_60.csv')
geo_pqr_23 = geo_pqr_23.drop(columns = ['Unnamed: 0'])
geo_pqr_23_uv = geo_pqr_23[['time', 'site_id', 'total_undervolt_events', 'total_undervolt_duration']]

# concatentate 
geo_pqr_uv_all = pd.concat([geo_pqr_22_uv, geo_pqr_23_uv], ignore_index=True)

In [ ]:
# Set undervolt_events to 0 where total_undervolt_events is 0 (no undervoltage for the specified duration)
geo_pqr_uv_all['unv_events_adj_60min'] = geo_pqr_uv_all['total_undervolt_events'].where(
    geo_pqr_uv_all['total_undervolt_events'] > 0, 0
)

geo_unv_60min = geo_pqr_uv_all.groupby('site_id')[['unv_events_adj_60min', 'total_undervolt_duration']].sum().reset_index()

geo_unv_60min = geo_unv_60min.rename(columns = {'unv_events_adj_60min':'60min_undervolt_events'})

## remove site '0'
geo_unv_60min = geo_unv_60min[~(geo_unv_60min['site_id'] == 0)]

# round down to nearest whole number 
float_cols = geo_unv_60min.select_dtypes(include='float').columns
geo_unv_60min[float_cols] = np.floor(geo_unv_60min[float_cols])

### Undervolts - 240_min 

In [ ]:
## 2022 
geo_pqr_22 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/Geospatial_Voltage_Durations/Files/voltage_hourly_22_und_240.csv')
geo_pqr_22 = geo_pqr_22.drop(columns = ['Unnamed: 0'])
geo_pqr_22_uv = geo_pqr_22[['time', 'site_id', 'total_undervolt_events', 'total_undervolt_duration']]

## 2023 
geo_pqr_23 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/Geospatial_Voltage_Durations/Files/voltage_hourly_23_und_240.csv')
geo_pqr_23 = geo_pqr_23.drop(columns = ['Unnamed: 0'])
geo_pqr_23_uv = geo_pqr_23[['time', 'site_id', 'total_undervolt_events', 'total_undervolt_duration']]

# concatenate 
geo_pqr_uv_all = pd.concat([geo_pqr_22_uv, geo_pqr_23_uv], ignore_index=True)

In [ ]:
# Set undervolt_events to 0 where total_undervolt_events is 0
geo_pqr_uv_all['unv_events_adj_240min'] = geo_pqr_uv_all['total_undervolt_events'].where(
    geo_pqr_uv_all['total_undervolt_events'] > 0, 0
)

geo_unv_240min = geo_pqr_uv_all.groupby('site_id')[['unv_events_adj_240min', 'total_undervolt_duration']].sum().reset_index()

geo_unv_240min = geo_unv_240min.rename(columns = {'unv_events_adj_240min':'240min_undervolt_events'})

## remove site '0'
geo_unv_240min = geo_unv_240min[~(geo_unv_240min['site_id'] == 0)]

# round down to nearest whole number 
float_cols = geo_unv_240min.select_dtypes(include='float').columns
geo_unv_240min[float_cols] = np.floor(geo_unv_240min[float_cols])

### Find average metrics for EAs with multiple sites 

In [ ]:
def compute_avg_values(site_list, df, event_col):
    geo_lookup = df.set_index('site_id')
    
    # Only use site_ids that are in geo_lookup
    valid_sites = [site for site in site_list if site in geo_lookup.index]
    
    if not valid_sites:
        return pd.Series({
            event_col: np.nan,
            # duration_col: np.nan
        })

    matching = geo_lookup.loc[valid_sites]
    avg_events = matching[event_col].mean()
    # avg_duration = matching[duration_col].mean()
    
    return pd.Series({
        event_col: avg_events,
        # duration_col: avg_duration
    })

In [ ]:
##### Outage Metrics ##### 

# copy of df 
grouped_sites_df_rez = grouped_sites_df.copy()


### 1+ hour 
averages_df = grouped_sites_df['site_id'].apply(
    lambda site_list: compute_avg_values(
        site_list,
        df = geo_1hr_grouped,
        event_col = '60min_outage_events',
    )
)
grouped_sites_df_rez = grouped_sites_df_rez.join(averages_df)

# -- # 

### 8+ hour 
averages_df = grouped_sites_df['site_id'].apply(
    lambda site_list: compute_avg_values(
        site_list,
        df = geo_8hr_grouped,
        event_col = '480min_outage_events',
    )
)
grouped_sites_df_rez = grouped_sites_df_rez.join(averages_df)

In [ ]:
##### Undervoltage Metrics ##### 


### 1+ hour 
averages_df = grouped_sites_df['site_id'].apply(
    lambda site_list: compute_avg_values(
        site_list,
        df = geo_unv_60min,
        event_col = '60min_undervolt_events',
    )
)
grouped_sites_df_rez = grouped_sites_df_rez.join(averages_df)

# -- # 

### 4+ hour 
averages_df = grouped_sites_df['site_id'].apply(
    lambda site_list: compute_avg_values(
        site_list,
        df = geo_unv_240min,
        event_col = '240min_undervolt_events',
    )
)
grouped_sites_df_rez = grouped_sites_df_rez.join(averages_df)

In [ ]:
## drop null values 
grouped_sites_df_rez = grouped_sites_df_rez.dropna()

### CESI and Average Grid Metrics per EA 

In [ ]:
## set ea_code as index, select the CESI column and tack on Average Grid Metrics per EA data 
if new_data_study_area.index.name != 'ea_code9ch':
    new_data_study_area = new_data_study_area.set_index('ea_code9ch')

new_data_study_area_copy = new_data_study_area.copy()[['climate_vul_add']]
new_data_study_area_copy = new_data_study_area_copy.join(grouped_sites_df_rez).dropna()

### Using K_Means to classify *CESI* into clusters (Low, Mid, High) 

In [ ]:
cesi_df = new_data_study_area_copy.copy()[['climate_vul_add']]

cesi_values = cesi_df[['climate_vul_add']].values

# Run KMeans

# random state (42) ensures you get the same results each time you run the code 
# n_init -> how many times to run the algorithm. using default state 
kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
cesi_df['cesi_cluster'] = kmeans.fit_predict(cesi_values)

# Map clusters to 'Low', 'Mid', 'High' based on cluster centers
# Sort cluster labels by increasing vulnerability
cluster_order = kmeans.cluster_centers_.flatten().argsort()
label_map = {cluster: label for cluster, label in zip(cluster_order, ['Low', 'Medium', 'High'])}
cesi_df['cesi_level'] = cesi_df['cesi_cluster'].map(label_map)

cesi_df = cesi_df.drop(columns=['cesi_cluster', 'climate_vul_add'])

In [ ]:
### Assign output of K_Means Clustering back to df 
new_data_study_area_copy = new_data_study_area_copy.join(cesi_df)


# Low (0.16 - 0.33); n = 167 
# Medium (0.34 - 0.48); n = 120 
# High (0.48 - 0.78); n = 50 

### Using K_Means to classify *Grid Metrics* into clusters (Low, Mid, High) 

In [ ]:
def add_outage_level_kmeans(
    df, 
    col='60min_outage_events', 
    n_clusters=3, 
    random_state=42,
    exclude_outlier_percentile=0.95  
):
    outage_df = df[[col]].copy()

    # Exclude top X percentile for clustering (outliers have unusually high values)
    if exclude_outlier_percentile is not None:
        threshold = outage_df[col].quantile(exclude_outlier_percentile)
        inlier_mask = outage_df[col] <= threshold
    else:
        inlier_mask = np.ones(len(outage_df), dtype=bool)

    # Fit KMeans only on inlier data
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init='auto')
    kmeans.fit(outage_df.loc[inlier_mask, [col]])

    # Predict cluster for all data (including outliers)
    outage_df['outage_cluster'] = kmeans.predict(outage_df[[col]])

    # Sort cluster labels by cluster center
    cluster_order = kmeans.cluster_centers_.flatten().argsort()
    label_map = {cluster: label for cluster, label in zip(cluster_order, ['Low', 'Medium', 'High'])}
    outage_df[f'{col}_level'] = outage_df['outage_cluster'].map(label_map)

    # Keep only the labeled column
    outage_level_df = outage_df[[f'{col}_level']]
    
    # Join back to the original DataFrame
    df_out = df.copy()
    df_out = df_out.join(outage_level_df)
    
    return df_out

### Run K-Means clustering on df  

In [ ]:
### Outages 

updated_study_area_df = add_outage_level_kmeans(new_data_study_area_copy, col='60min_outage_events', exclude_outlier_percentile=0.95)
updated_study_area_df = add_outage_level_kmeans(updated_study_area_df, col='480min_outage_events', exclude_outlier_percentile=0.95)

### Undervoltages  

updated_study_area_df = add_outage_level_kmeans(updated_study_area_df, col='60min_undervolt_events', exclude_outlier_percentile=0.95)
updated_study_area_df = add_outage_level_kmeans(updated_study_area_df, col='240min_undervolt_events', exclude_outlier_percentile=0.95)

### Generate Bins for Grid Metrics 

In [ ]:
def get_outage_bin_edges(df, value_col, level_col):

    levels = df.groupby(level_col)[value_col].agg(['min', 'max'])
    
    # Ensure consistent order
    levels = levels.loc[['Low', 'Medium', 'High']]

    # Create bin edges: min of Low, min of Medium, min of High, and inf
    bin_edges = [
        levels.loc['Low', 'min'],
        levels.loc['Medium', 'min'],
        levels.loc['High', 'min'],
        np.inf
    ]
    return bin_edges

In [ ]:
### Outages 

bins_outage_60min = get_outage_bin_edges(
    updated_study_area_df,
    value_col='60min_outage_events',
    level_col='60min_outage_events_level'
)

bins_outage_480min = get_outage_bin_edges(
    updated_study_area_df,
    value_col='480min_outage_events',
    level_col='480min_outage_events_level'
)

In [ ]:
### Undervoltages

bins_undervolt_60min = get_outage_bin_edges(
    updated_study_area_df,
    value_col='60min_undervolt_events',
    level_col='60min_undervolt_events_level'
)

bins_undervolt_240min = get_outage_bin_edges(
    updated_study_area_df,
    value_col='240min_undervolt_events',
    level_col='240min_undervolt_events_level'
)

In [ ]:
## Generate legend labels for the legend in the bivariate plot 

def generate_bin_labels(bin_edges):
    labels = []
    for i in range(len(bin_edges) - 1):
        left = round(bin_edges[i])
        right = bin_edges[i + 1]

        if np.isinf(right):
            labels.append(f"{left}+")
        else:
            right_label = round(right)
            labels.append(f"{left}-{right_label}")
    return labels

## Bivariate Chloropleth Maps Workflow 

In [ ]:
def plot_bivar_map(
    df, 
    y_bins, 
    district_gdf, 
    title='1+hr Outage Events & Climate Vulnerability', 
    y_label='1+hour Outage Events', 
    save_path=None, 
    xlim=None, 
    ylim=None,
    show_legend=True,       
    add_basemap=False,        
    basemap_source=ctx.providers.OpenStreetMap.Mapnik  
):
    fig, ax = plt.subplots(figsize=(12, 10), dpi=300)

    # Plot bivariate classes
    df.plot(ax=ax, column='Bi_Class', cmap=cmap, categorical=True, legend=False)
    
    # Plot all EAs
    filt_all_eas_dem_df_filt.plot(ax=ax, facecolor='None', edgecolor='dimgrey', lw=0.4, alpha = 0.3)
    # filt_all_eas_dem_df_filt.plot(ax=ax, facecolor='None', edgecolor='white', lw=0.4, alpha = 0.3)

    # Plot district boundary
    district_gdf.plot(ax=ax, facecolor='None', edgecolor='k', lw=1.2)
    
    # Apply zoom limits if provided
    if xlim:
        ax.set_xlim(xlim)
    if ylim:
        ax.set_ylim(ylim)

    # === Optional Basemap ===
    if add_basemap:
        ctx.add_basemap(ax, crs=df.crs, source=basemap_source, attribution=None)

    ax.set_axis_off()
    ax.set_title(f'{title}', fontsize=14, pad=8)

    # === Optional Legend ===
    if show_legend:
        ax2 = fig.add_axes([0.75, 0.18, 0.12, 0.12])
        alpha = 1

        # Draw 3x3 grid
        for i, (x, y) in enumerate([(x, y) for y in range(3) for x in range(3)]):
            ax2.axvspan(
                xmin=x/3, xmax=(x+1)/3,
                ymin=y/3, ymax=(y+1)/3,
                alpha=alpha, color=colors[i]
            )

        # Ticks and labels
        ax2.set_xticks([0.165, 0.5, 0.835])
        ax2.set_yticks([0.165, 0.5, 0.835])
        ax2.set_xlim(0, 1)
        ax2.set_ylim(0, 1)
        ax2.set_aspect('equal')

        # Axis labels
        ax2.set_xticklabels(generate_bin_labels(y_bins), rotation=60, ha='center')  # Grid metric labels
        ax2.set_yticklabels(['Low (0.16–0.33)', 'Mid (0.34–0.48)', 'High (0.48–0.78)'])  # CESI labels
        ax2.set_xlabel(f"{y_label}", labelpad=8, fontsize=10, fontweight='bold')
        ax2.set_ylabel("CESI", labelpad=8, fontsize=10, fontweight='bold')
        ax2.tick_params(axis='both', which='both', length=4, width=1, labelsize=9)

    # === Save or Show ===
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
### Zoomed version 

def plot_bivar_map_zoomed(
                    df, 
                    y_bins, 
                    district_gdf, 
                   title='1+hr Outage Events & Climate Vulnerability', 
                   y_label='1+hour Outage Events', 
                   fig_size=(12, 10), 
                   save_path=None, 
                   xlim=None, 
                   ylim=None, 
                   add_basemap=False,   
                   basemap_source=ctx.providers.OpenStreetMap.Mapnik,   
                   show_legend=False):

    fig, ax = plt.subplots(figsize=fig_size, dpi=300)

    # Plot bivariate classes
    df.plot(ax=ax, column='Bi_Class', cmap=cmap, categorical=True, legend=False)

    
    # Plot all EAs
    filt_all_eas_dem_df_filt.plot(ax=ax, facecolor='None', edgecolor='dimgrey', lw=0.4, alpha = 0.3)
    # filt_all_eas_dem_df_filt.plot(ax=ax, facecolor='None', edgecolor='white', lw=0.4, alpha = 0.3)

    # Plot district boundary
    district_gdf.plot(ax=ax, facecolor='None', edgecolor='k', lw=1.2)


    # Apply axis limits if provided
    if xlim:
        ax.set_xlim(xlim)
    if ylim:
        ax.set_ylim(ylim)

     # === Optional Basemap ===
    if add_basemap:
        ctx.add_basemap(ax, crs=df.crs, source=basemap_source, attribution=None)

    ax.set_axis_off()
    ax.set_title(f'{title}', fontsize=14, pad=8)

    # === Optional Legend ===
    if show_legend:
        ax2 = fig.add_axes([0.78, 0.27, 0.08, 0.08])
        alpha = 1

        # Draw 3x3 grid
        for i, (x, y) in enumerate([(x, y) for y in range(3) for x in range(3)]):
            ax2.axvspan(
                xmin=x/3, xmax=(x+1)/3,
                ymin=y/3, ymax=(y+1)/3,
                alpha=alpha, color=colors[i]
            )

        # Axis settings for legend
        ax2.set_xticks([0.165, 0.5, 0.835])
        ax2.set_yticks([0.165, 0.5, 0.835])
        ax2.set_xlim(0, 1)
        ax2.set_ylim(0, 1)
        ax2.set_aspect('equal')


        ax2.set_xticklabels(generate_bin_labels(y_bins), rotation=60, ha='center')
        ax2.set_yticklabels(['Low (0.16-0.33)', 'Mid (0.34-0.48)', 'High (0.48-0.78)'])
        ax2.set_xlabel(f"{y_label}", labelpad=8, fontsize=8, fontweight='bold')
        ax2.set_ylabel("CESI", labelpad=8, fontsize=8, fontweight='bold')
        ax2.tick_params(axis='both', which='both', length=4, width=1, labelsize=8)

    # === Save or Show ===
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

### 1+hr outage events - K_Means 

In [ ]:
## **** copy of UPDATED STUDY AREA  
bivariate_df = updated_study_area_df.copy()

### get geometry 
new_data_study_area_geom = new_data_study_area[['geometry']]

bivariate_df = new_data_study_area_geom.join(bivariate_df)

# Define code mappings
x_code_map = {'Low': '1', 'Medium': '2', 'High': '3'}
y_code_map = {'Low': 'A', 'Medium': 'B', 'High': 'C'}

#### Select the column here ----> 

# Assign x and y classes based on existing labels
bivariate_df['Var1_Class'] = bivariate_df['60min_outage_events_level'].map(x_code_map)
bivariate_df['Var2_Class'] = bivariate_df['cesi_level'].map(y_code_map)
bivariate_df = bivariate_df.dropna(subset=['Var1_Class', 'Var2_Class'])

# Column for Bivariate Class 
bivariate_df['Bi_Class'] = (
    bivariate_df['Var2_Class'].astype(str) + bivariate_df['Var1_Class'].astype(str)
)
bivariate_df['Bi_Class'] = bivariate_df['Bi_Class'].astype('category')

In [ ]:
colors = [
    '#6a51a3',  # A1 - dark purple
    '#9e9ac8',  # A2 - muted purple
    '#f2f0f7',  # A3 - very light purple
    '#9e9ac8',  # B1 - muted purple
    '#f2f0f7',  # B2 - pale neutral
    '#fdae6b',  # B3 - soft brown-orange
    '#f2f0f7',  # C1 - pale neutral
    '#fdae6b',  # C2 - brown-orange
    '#8c2d04'   # C3 - dark brown (high)
]
cmap = matplotlib.colors.ListedColormap(colors)

In [ ]:
colors = [
    '#542788',  # A1 - dark purple (low-low)
    '#8073ac',  # A2 - medium purple
    '#b2abd2',  # A3 - light purple
    '#d8daeb',  # B1 - pale purple-grey
    '#f7f7f7',  # B2 - neutral (center)
    '#fee0b6',  # B3 - pale tan
    '#fdb863',  # C1 - light orange
    '#e08214',  # C2 - medium orange
    '#b35806'   # C3 - dark brown (high-high)
]

cmap = matplotlib.colors.ListedColormap(colors)

In [ ]:
plot_bivar_map(bivariate_df, 
               y_bins = bins_outage_60min, 
               district_gdf = dist_2021_transformed, 
               title = '', 
               y_label = '1+hour Outage Events', 
               add_basemap=False, 
               # save_path = '/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/images/bivar_NEW_District.png'
              )

In [ ]:
### Zoomed 

plot_bivar_map_zoomed(
                bivariate_df, 
               bins_outage_60min, 
               district_gdf = dist_2021_transformed, 
               title = '', 
               y_label = '1+hour Outage Events', 
               xlim = (798500, 812200), 
               ylim = (610000,621000), 
               add_basemap=False, 
               fig_size=(7,5), 
               save_path = '/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/images/ZOOM_district_1hr.png'
              )

### 8+hr outage events - K_Means 

In [ ]:
## **** copy of UPDATED STUDY AREA  
bivariate_df = updated_study_area_df.copy()

### get geometry 
new_data_study_area_geom = new_data_study_area[['geometry']]

bivariate_df = new_data_study_area_geom.join(bivariate_df)

# Define code mappings
x_code_map = {'Low': '1', 'Medium': '2', 'High': '3'}
y_code_map = {'Low': 'A', 'Medium': 'B', 'High': 'C'}

#### Select the column here ----> 

# Assign x and y classes based on existing labels
bivariate_df['Var1_Class'] = bivariate_df['480min_outage_events_level'].map(x_code_map)
bivariate_df['Var2_Class'] = bivariate_df['cesi_level'].map(y_code_map)
bivariate_df = bivariate_df.dropna(subset=['Var1_Class', 'Var2_Class'])

# Column for Bivariate Class 
bivariate_df['Bi_Class'] = (
    bivariate_df['Var2_Class'].astype(str) + bivariate_df['Var1_Class'].astype(str)
)
bivariate_df['Bi_Class'] = bivariate_df['Bi_Class'].astype('category')

In [ ]:
# plot_bivar_map(bivariate_df, 
#                bins_outage_480min, 
#                district_gdf = dist_2021_transformed, 
#                title = '', 
#                y_label = '8+hour Outage Events', 
#                add_basemap=False, 
#               save_path = '/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/images/bivar_8hr_k_means.png'
#               )

In [ ]:
### Zoomed 

# plot_bivar_map_zoomed(
#                bivariate_df, 
#                bins_outage_480min, 
#                district_gdf = dist_2021_transformed, 
#                title = '', 
#                y_label = '8+hour Outage Events', 
#                xlim = (798500, 812200), 
#                ylim = (610000,621000), 
#                fig_size=(7,5), 
#                show_legend = False, 
#                save_path = '/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/images/bivar_ZOOMED_8hr_inset_2.png'
#               )

### 60min undervolt events - K_Means

In [ ]:
## **** copy of UPDATED STUDY AREA  
bivariate_df = updated_study_area_df.copy()

### get geometry 
new_data_study_area_geom = new_data_study_area[['geometry']]

bivariate_df = new_data_study_area_geom.join(bivariate_df)

# Define code mappings
x_code_map = {'Low': '1', 'Medium': '2', 'High': '3'}
y_code_map = {'Low': 'A', 'Medium': 'B', 'High': 'C'}

#### Select the column here ----> 

# Assign x and y classes based on existing labels
bivariate_df['Var1_Class'] = bivariate_df['60min_undervolt_events_level'].map(x_code_map)
bivariate_df['Var2_Class'] = bivariate_df['cesi_level'].map(y_code_map)
bivariate_df = bivariate_df.dropna(subset=['Var1_Class', 'Var2_Class'])

# Column for Bivariate Class 
bivariate_df['Bi_Class'] = (
    bivariate_df['Var2_Class'].astype(str) + bivariate_df['Var1_Class'].astype(str)
)
bivariate_df['Bi_Class'] = bivariate_df['Bi_Class'].astype('category')

In [ ]:
plot_bivar_map(bivariate_df, 
               bins_undervolt_60min, 
               title = '', 
              district_gdf = dist_2021_transformed, 
               y_label = '1+hour Undervolts', 
              save_path = '/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/images/bivar_1hr_undervolt_k_means.png'
              )

In [ ]:
plot_bivar_map_zoomed(bivariate_df, 
               bins_undervolt_60min, 
               title = '', 
               district_gdf = dist_2021_transformed, 
               y_label = '1+hour Undervolts', 
               xlim = (798500, 812200), 
               ylim = (610000,621000), 
               fig_size=(7,5), 
               show_legend = False, 
               save_path = '/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/images/bivar_ZOOMED_unv_1hr_inset.png'
              )

### 240min undervolt events - K_Means

In [ ]:
## **** copy of UPDATED STUDY AREA  
bivariate_df = updated_study_area_df.copy()

### get geometry 
new_data_study_area_geom = new_data_study_area[['geometry']]

bivariate_df = new_data_study_area_geom.join(bivariate_df)

# Define code mappings
x_code_map = {'Low': '1', 'Medium': '2', 'High': '3'}
y_code_map = {'Low': 'A', 'Medium': 'B', 'High': 'C'}

#### Select the column here ----> 

# Assign x and y classes based on existing labels
bivariate_df['Var1_Class'] = bivariate_df['240min_undervolt_events_level'].map(x_code_map)
bivariate_df['Var2_Class'] = bivariate_df['cesi_level'].map(y_code_map)
bivariate_df = bivariate_df.dropna(subset=['Var1_Class', 'Var2_Class'])

# Column for Bivariate Class 
bivariate_df['Bi_Class'] = (
    bivariate_df['Var2_Class'].astype(str) + bivariate_df['Var1_Class'].astype(str)
)
bivariate_df['Bi_Class'] = bivariate_df['Bi_Class'].astype('category')

In [ ]:
plot_bivar_map(bivariate_df, 
               bins_undervolt_240min, 
               title = '', 
               district_gdf = dist_2021_transformed, 
               y_label = '4+hour Undervolts', 
               show_legend = True, 
              save_path = '/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/images/bivar_4hr_undervolt_k_means.png'
              )

In [ ]:
plot_bivar_map_zoomed(bivariate_df, 
               bins_undervolt_240min, 
               title = '', 
               district_gdf = dist_2021_transformed, 
               y_label = '4+hour Undervolts', 
               xlim = (798500, 812200), 
               ylim = (610000,621000), 
               fig_size=(7,5), 
               show_legend = False, 
               save_path = '/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/images/bivar_ZOOMED_unv_4hr_inset.png'
              )

## *** Spatial Co_occurrence Maps *** 

### Outages - 1+ hour (TPL dataset)

In [ ]:
# read file from spatial co-occurrence analysis 
outage_1hr_path = '/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/rez_spatial_1hr_TPL.csv'
outage_1hr_spatial = pd.read_csv(outage_1hr_path).drop(columns = ['Unnamed: 0']).set_index('ea_code9ch')

## 95th percentile 
outage_1hr_spatial_95 = outage_1hr_spatial[outage_1hr_spatial['Pct'] == '95th']
outage_1hr_spatial_95 = new_data_study_area[['climate_vul_add', 'geometry']].join(outage_1hr_spatial_95).dropna()

In [ ]:
def plot_spatial_co_occur_map(
    gdf,  
    district_gdf, 
    title='', 
    column='PL_outage_prob', 
    cmap = 'Blues', 
    save_path=None,
    n_bins=5,                
    xlim=None,               
    ylim=None,               
    x_label=None,            
    fig_size=(12, 10),
    add_basemap=False,        
    basemap_source=ctx.providers.OpenStreetMap.Mapnik,   
    show_legend=True, 
    hide_attribution=True,
    legend_fraction=0.03,    
    legend_pad=0.01,         
    legend_aspect=25,        
    legend_orientation='horizontal'  # NEW: 'horizontal' or 'vertical'
):

    gdf = gdf.copy()
    
    fig, ax = plt.subplots(figsize=fig_size, dpi=300)

    # Convert to percentage if values are 0–1
    if gdf[column].max() <= 1:
        gdf[column] = gdf[column] * 100
    
    # Define discrete bins
    bins = list(np.linspace(0, 100, n_bins+1))
    cmap = plt.colormaps.get_cmap(cmap).resampled(n_bins)
    norm = mcolors.BoundaryNorm(bins, cmap.N)

    # Plot main data
    gdf.plot(
        ax=ax,
        column=column,
        cmap=cmap,
        norm=norm,
        legend=False
    )

    # Plot all EAs
    filt_all_eas_dem_df_filt.plot(ax=ax, facecolor='None', edgecolor='dimgrey', lw=0.4, alpha=0.3)

    # Plot district boundary
    district_gdf.plot(ax=ax, facecolor='None', edgecolor='k', lw=1.2)

    # Apply map zoom limits if provided
    if xlim:
        ax.set_xlim(xlim)
    if ylim:
        ax.set_ylim(ylim)

    # Optional basemap
    if add_basemap:
        ctx.add_basemap(ax, crs=gdf.crs, source=basemap_source, attribution=None)

    ax.set_axis_off()
    ax.set_title(title, fontsize=14, pad=8)

    # === Add Colorbar (Legend) ===
    if show_legend:
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])

        # dynamic legend
        cbar = fig.colorbar(
            sm,
            ax=ax,
            orientation=legend_orientation,
            fraction=legend_fraction,
            pad=legend_pad,
            aspect=legend_aspect,
            ticks=bins
        )

        # Format tick labels as %
        cbar.set_ticks(bins)
        cbar.set_ticklabels([f"{int(b)}%" for b in bins])

        # 🔹 Increase legend tick size
        cbar.ax.tick_params(labelsize=14)

        # Label positioning
        if x_label:
            cbar.set_label(x_label, fontsize=14, fontweight='bold', labelpad=12)
        else:
            cbar.set_label(f"{column} (%)", fontsize=14, fontweight='bold', labelpad=12)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()

In [ ]:
plot_spatial_co_occur_map(
    outage_1hr_spatial_95, 
    dist_2021_transformed, 
    cmap = 'viridis', 
    column='PL_outage_prob', 
    # x_label='Probability of PL-Outages (%)', 
    x_label='PL-Outage (1+hour) Probability (%)', 
    add_basemap=False, 
    legend_orientation='horizontal', 
    legend_aspect=50, 
    save_path='/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/spatial_co_occurrence/1hr_PL_outage_prob.png'
)

In [ ]:
### Zoomed for Inset 

plot_spatial_co_occur_map(
    outage_1hr_spatial_95,  
    dist_2021_transformed, 
    cmap = 'viridis', 
    title='', 
    column='PL_outage_prob', 
    xlim = (798500, 812200), 
    ylim = (610000,621000),
    fig_size=(7, 5),
    show_legend=False, 
    save_path='/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/spatial_co_occurrence/1hr_PL_outage_prob_spatial_plot_zoomed.png'
)

### EAs with >__% probability 

In [ ]:
# thr = 0.3
# thr = 0.5
thr = 0.8

mask = outage_1hr_spatial_95["PL_outage_prob"] >= thr
count = mask.sum()
percent = mask.mean() * 100

print(f"Number of EAs with outage probability >= {thr}: {count}")
print(f"Percentage of EAs with outage probability >= {thr}: {percent:.2f}%")

### Scatter Plot  

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import statsmodels.api as sm

def plot_cesi_vs_grid_disturbance(df, 
                        x_col='climate_vul_add',   
                        y_col='PL_outage_prob',   
                        fig_size=(8, 6), 
                        size=30, 
                        x_label='Climate Exposure & Sensitivity Index', 
                        y_label='PL-Outage (1+ hour) Probability',
                        add_regression=True, 
                        legend_loc='center right',    
                        legend_coords=(1.00, 0.5), 
                        save_path = None
                        ):

    fig, ax = plt.subplots(figsize=fig_size, dpi=300)

    # --- Categorize climate vulnerability levels ---
    bins = [0, 0.33, 0.48, df[x_col].max()+0.01]
    
    # labels = ['Low CESI (0.16 – 0.33)', 'Medium CESI (0.34 – 0.48)', 'High CESI (0.48 - 0.78)']
    labels = ['Low CESI', 'Medium CESI', 'High CESI']

    colors = [
              '#6a51a3', 
              '#f2f0f7', 
              '#8c2d04'  
                ]  

    df['vul_category'] = pd.cut(df[x_col], bins=bins, labels=labels, include_lowest=True)

    # --- Scatter plot for each category ---
    for cat, color in zip(labels, colors):
        subset = df[df['vul_category'] == cat]
        ax.scatter(
            subset[x_col],
            subset[y_col],
            alpha=0.7,
            edgecolor='k',
            lw=0.5,
            s=size,
            color=color,
            label=cat
        )

    # --- Linear regression ---
    if add_regression:
        X = sm.add_constant(df[x_col])
        model = sm.OLS(df[y_col], X).fit()
        intercept, slope = model.params

        x_vals = np.linspace(df[x_col].min(), df[x_col].max(), 100)
        X_pred = sm.add_constant(x_vals)
        pred = model.get_prediction(X_pred)
        y_pred = pred.predicted_mean
        ci_lower, ci_upper = pred.conf_int().T

        # Pearson correlation
        r_value = df[x_col].corr(df[y_col])

        # Plot regression line and shaded 95% CI
        ax.plot(x_vals, y_pred, color='black', linewidth=2, 
                label=f'Linear fit (r = {r_value:.2f})')
        ax.fill_between(x_vals, ci_lower, ci_upper, color='grey', alpha=0.2, label='95% Confidence')

    # --- Labels and formatting ---
    ax.set_xlabel(x_label, fontsize=12, fontweight='bold', labelpad = 15)
    ax.set_ylabel(y_label, fontsize=12, fontweight='bold', labelpad = 15)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

    # --- Legend and style ---
    ax.legend(loc=legend_loc, bbox_to_anchor=legend_coords, borderaxespad=0.)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        
    plt.show()

In [ ]:
plot_cesi_vs_grid_disturbance(
    outage_1hr_spatial_95, 
    size=20, 
    add_regression=True, 
    save_path='/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/spatial_co_occurrence/PL_outage_scatter_plot.png'
)

## 8+ hour Outages 

In [ ]:
outage_8hr_path = '/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/rez_spatial_8hr_TPL.csv'
outage_8hr_spatial = pd.read_csv(outage_8hr_path).drop(columns = ['Unnamed: 0']).set_index('ea_code9ch')

## 95th percentile 
outage_8hr_spatial_95 = outage_8hr_spatial[outage_8hr_spatial['Pct'] == '95th']
outage_8hr_spatial_95 = new_data_study_area[['climate_vul_add', 'geometry']].join(outage_8hr_spatial_95).dropna()

In [ ]:
# plot_spatial_co_occur_map(
#     outage_8hr_spatial_95, 
#     dist_2021_transformed, 
#     cmap = 'viridis', 
#     column='PL_outage_prob', 
#     x_label='Probability of PL - Outages (%)', 
#     add_basemap=False, 
#     legend_orientation='horizontal', 
#     legend_aspect=40, 
#     # save_path='/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/spatial_co_occurrence/8hr_PL_outage_prob.png'
# )

In [ ]:
### Zoomed version 

# plot_spatial_co_occur_map(
#     outage_8hr_spatial_95,  
#     dist_2021_transformed, 
#     cmap = 'viridis', 
#     title='', 
#     column='PL_outage_prob', 
#     xlim = (798500, 812200), 
#     ylim = (610000,621000),
#     fig_size=(7, 5),
#     show_legend=False, 
#     # save_path='/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/spatial_co_occurrence/8hr_PL_outage_prob_spatial_plot_zoomed.png'
# )

In [ ]:
### Scatter plot 

# plot_cesi_vs_grid_disturbance(
#     outage_8hr_spatial_95, 
#     size=20, 
#     add_regression=True)

## Undervoltages (TPL dataset)

In [ ]:
# read file from spatial co-occurrence analysis 
undervolt_4hr_path = '/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/rez_SPATIAL_unv_4hr_TPL_rev.csv'
undervolt_4hr_spatial = pd.read_csv(undervolt_4hr_path).drop(columns = ['Unnamed: 0']).set_index('ea_code9ch')

## using the 95th percentile 
undervolt_4hr_spatial_95 = undervolt_4hr_spatial[undervolt_4hr_spatial['Pct'] == '95th']
undervolt_4hr_spatial_95 = new_data_study_area[['climate_vul_add', 'geometry']].join(undervolt_4hr_spatial_95).dropna()

In [ ]:
plot_spatial_co_occur_map(
    undervolt_4hr_spatial_95,  
    dist_2021_transformed, 
    cmap = 'viridis', 
    title='', 
    column='T_undervolt_prob', 
    # x_label='Probability of T-Undervoltages (%)', 
    x_label='T-Undervoltage (4+hour) Probability (%)', 
    add_basemap=False, 
    legend_orientation='horizontal',    
    legend_aspect=50, 
    # save_path='/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/spatial_co_occurrence/4hr_T_undervolt_prob_rev1.png'
)

In [ ]:
### Zoomed version 

plot_spatial_co_occur_map(
    undervolt_4hr_spatial_95,  
    dist_2021_transformed, 
    cmap = 'viridis', 
    title='', 
    column='T_undervolt_prob', 
    xlim = (798500, 812200), 
    ylim = (610000,621000),
    fig_size=(7, 5),
    show_legend=False, 
    # save_path='/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/spatial_co_occurrence/4hr_T_undervolt_prob_zoomed.png'
)

### EAs with >__% probability 

In [ ]:
# thr = 0.5
# thr = 0.6
thr = 0.8

mask = undervolt_4hr_spatial_95["T_undervolt_prob"] >= thr
count = mask.sum()
percent = mask.mean() * 100

print(f"Number of EAs with undervolt probability >= {thr}: {count}")
print(f"Percentage of EAs with undervolt probability >= {thr}: {percent:.2f}%")

### Scatter Plot  

In [ ]:
plot_cesi_vs_grid_disturbance(
    undervolt_4hr_spatial_95, 
    x_col='climate_vul_add',   
    y_col='T_undervolt_prob', 
    y_label='T-Undervoltage (4+hour) Probability',
    size=20, 
    add_regression=True, 
    legend_coords=(1.02, 0.6), 
    save_path='/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/spatial_co_occurrence/T_undervolt_scatter_plot.png'
)